# SimSiam CIFAR-10 Baseline

Existing SimSiam model trained on the shared N-pool with the suite LARS recipe, followed by diagnostics.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('google.colab is only available inside Google Colab; skipping Drive mount.')


In [ ]:
import getpass
import os
import subprocess
from pathlib import Path
from urllib.parse import quote

PUBLIC_REPO_URL = 'https://github.com/moucheng2017/instance_self_sup.git'
BRANCH = '2-vicreg-barlow-twins-baselines'  # Use 'main' after this baseline suite is merged.
REPO_DIR = Path('/content/instance_self_sup')
GITHUB_TOKEN = ''  # Set to a token string here if the repo is private.


def run(cmd, cwd=None):
    print('+', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


try:
    from google.colab import userdata
    GITHUB_TOKEN = GITHUB_TOKEN or userdata.get('GITHUB_TOKEN')
except Exception:
    pass

repo_url = PUBLIC_REPO_URL
if GITHUB_TOKEN:
    repo_url = PUBLIC_REPO_URL.replace('https://', f'https://oauth2:{quote(GITHUB_TOKEN, safe="")}@')


def sync_repo():
    if GITHUB_TOKEN:
        run(['git', 'remote', 'set-url', 'origin', repo_url], cwd=REPO_DIR)
    else:
        print('No GitHub token found; keeping the existing origin URL for fetch/pull.')
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR)
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR)

os.chdir('/content')

if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', BRANCH, repo_url, str(REPO_DIR)])
else:
    sync_repo()

os.chdir(REPO_DIR)
run(['python', '-m', 'pip', 'install', '-r', 'requirements_colab.txt'])
print('Repository is ready at', REPO_DIR)


In [ ]:
# CIFAR-10 is downloaded automatically by torchvision the first time this cell runs.
import os
import torchvision

DATA_DIR = '/content/drive/MyDrive/SSL_exps/data'
os.makedirs(DATA_DIR, exist_ok=True)

print('Checking / downloading CIFAR-10 ...')
torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True)
torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True)
print('CIFAR-10 is ready.')


## Sweep Parameters

| Parameter | Description |
|---|---|
| `N_SWEEP`, `subset_n`, `subset_seed` | Shared low-data pool for training and diagnostics |
| `DEFAULT_BATCH_SIZE`, `NUM_EPOCHS`, `STOP_AT_EPOCH` | Per-N training controls; batch size is clamped to `N` |
| `SAVING_FREQUENCY` | Save a checkpoint every N epochs; checkpoint filename includes the epoch number |
| `ITERATIONS_PER_SAMPLE`, `samples_per_epoch` | Per-epoch sample budget; matched to pseudo_sup so all methods run equal iterations/epoch per pool size |
| `INIT_CHECKPOINT`, `INIT_LOAD_BACKBONE`, `INIT_LOAD_PROJECTOR`, `INIT_LOAD_PREDICTOR` | Optional checkpoint initialization controls |
| `knn_monitor` / `monitor_accuracy` | In-training monitor is disabled by default; Section 2.1 KNN is computed in the diagnostics cell |
| `proj_layers` | SimSiam projector depth; default 3-layer MLP matches pseudo_sup's projector for fair comparison |


In [ ]:
from colab_utils import train_from_colab

import os

# --- Weights & Biases setup (mirrors the GITHUB_TOKEN pattern above) ---
# Easiest: add your key to Colab Secrets (left sidebar key icon) named WANDB_API_KEY,
# or paste it below. Leave blank to run without W&B.
WANDB_API_KEY = ''
WANDB_PROJECT = 'instance_self_sup'
WANDB_ENTITY = None  # your W&B team/username, or None for your default account
try:
    from google.colab import userdata
    WANDB_API_KEY = WANDB_API_KEY or userdata.get('WANDB_API_KEY')
except Exception:
    pass
USE_WANDB = bool(WANDB_API_KEY)
if USE_WANDB:
    os.environ['WANDB_API_KEY'] = WANDB_API_KEY
    try:
        import wandb
        wandb.login()
        print('W&B login OK.')
    except Exception as exc:
        USE_WANDB = False
        print('W&B login failed; continuing without W&B:', exc)
else:
    print('No WANDB_API_KEY found; W&B logging disabled for this run.')

CONFIG_FILE = 'configs/baselines/simsiam_cifar10.yaml'  # config_file='configs/baselines/simsiam_cifar10.yaml'
PROJECT_NAME = 'instance_self_sup'
SUBSET_SEED = 42
N_SWEEP = [200, 500, 1000, 5000] # 'full' means using the full dataset (50k training examples)
DEFAULT_BATCH_SIZE = 256
NUM_EPOCHS = 100
STOP_AT_EPOCH = 100
SAVING_FREQUENCY = 100  # save a checkpoint every N epochs; filename includes the epoch number

INIT_CHECKPOINT = None
INIT_LOAD_BACKBONE = True
INIT_LOAD_PROJECTOR = False
INIT_LOAD_PREDICTOR = False

# Full 3-layer projection MLP (512->2048->2048), matching pseudo_supervised_net's
# projector for a fair comparison. None also keeps this default; 3 is set explicitly
# to make projector-on parity unambiguous (set_layers accepts only 2 or 3).
PROJ_LAYERS = 3


def batch_size_for_n(n):
    if n == 'full' or n is None:
        return DEFAULT_BATCH_SIZE
    return min(DEFAULT_BATCH_SIZE, int(n))


# Iteration parity with pseudo_sup: match its per-epoch sample budget so every method
# runs the same number of iterations per epoch at each pool size. pseudo_sup uses
# samples_per_epoch = (pool_size / 0.25) * 20; we replicate that here (= pool_size * 80).
ITERATIONS_PER_SAMPLE = 20
POOL_TO_SAMPLES_FACTOR = 4  # = 1 / pseudo_sup negatives_ratio (0.25)


def samples_per_epoch_for_n(n):
    pool_size = 50000 if n == 'full' or n is None else int(n)
    batch_size = batch_size_for_n(n)
    return max(pool_size * POOL_TO_SAMPLES_FACTOR * ITERATIONS_PER_SAMPLE, int(pool_size // batch_size))


results = []
for n in N_SWEEP:
    subset_n = None if n == 'full' else int(n)
    batch_size = batch_size_for_n(n)
    samples_per_epoch = samples_per_epoch_for_n(n)
    overrides = {
        'name': f'{CONFIG_FILE.split("/")[-1].replace(".yaml", "")}-N{n}',
        'logger': {
            'wandb': USE_WANDB,
            'wandb_project': WANDB_PROJECT,
            'wandb_entity': WANDB_ENTITY,
        },
        'train': {
            'subset_n': subset_n,
            'subset_seed': SUBSET_SEED,
            'batch_size': batch_size,
            'num_epochs': NUM_EPOCHS,
            'stop_at_epoch': STOP_AT_EPOCH,
            'saving_frequency': SAVING_FREQUENCY,
            'samples_per_epoch': samples_per_epoch,
            'knn_monitor': False,
        },
        'model': {
            'init_checkpoint': INIT_CHECKPOINT,
            'init_load_backbone': INIT_LOAD_BACKBONE,
            'init_load_projector': INIT_LOAD_PROJECTOR,
            'init_load_predictor': INIT_LOAD_PREDICTOR,
        'proj_layers': PROJ_LAYERS,
        },
    }
    run_result = train_from_colab(
        config_file=CONFIG_FILE,
        project_name=PROJECT_NAME,
        overrides=overrides,
        device='cuda',
        download=True,
        hide_progress=False,
    )
    results.append({
        'N': n,
        'subset_n': subset_n,
        'subset_seed': SUBSET_SEED,
        'batch_size': batch_size,
        'model_path': run_result['model_path'],
        'log_dir': run_result['log_dir'],
        'selected_subset_indices_path': run_result.get('selected_subset_indices_path'),
        'monitor_accuracy': run_result.get('accuracy'),
    })

results


In [ ]:
import json
import os
import re

import matplotlib.pyplot as plt
import pandas as pd
import torch

from analysis.spectral import build_diagnostics_loader, extract_features, knn_eval, spectral_diagnostics
from arguments import build_args
from datasets.subset import load_subset_indices, select_subset_indices
from linear_eval import load_backbone_weights
from models import get_backbone


def selected_indices_for_result(args, item):
    selected_path = item.get('selected_subset_indices_path')
    if selected_path:
        return load_subset_indices(selected_path)
    if item['subset_n'] is None:
        return None
    probe_loader = build_diagnostics_loader(args, indices=None, batch_size=item['batch_size'])
    return select_subset_indices(len(probe_loader.dataset), item['subset_n'], item['subset_seed'])


def epoch_from_checkpoint(path):
    match = re.search(r'_epoch(\d+)_', os.path.basename(path))
    return int(match.group(1)) if match else -1


def checkpoints_for_result(item):
    """Every checkpoint saved for this run (one per saving_frequency interval), sorted by epoch.

    Reads <log_dir>/checkpoint_path.txt, which save_checkpoint appends to for each saved
    checkpoint; falls back to the final model_path if the listing is unavailable.
    """
    listing = os.path.join(item['log_dir'], 'checkpoint_path.txt')
    paths = []
    if os.path.exists(listing):
        with open(listing) as f:
            paths = [line.strip() for line in f if line.strip()]
    if not paths and item.get('model_path'):
        paths = [item['model_path']]
    unique = sorted(set(paths), key=epoch_from_checkpoint)
    return [p for p in unique if os.path.exists(p)]


def save_checkpoint_figure(n, epoch, singular_values, log_dir):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(range(1, len(singular_values) + 1), singular_values, marker='o')
    ax.set_xlabel('Singular value rank')
    ax.set_ylabel('Singular value')
    ax.set_title(f'Top-20 penultimate-feature singular values (N={n}, epoch {epoch})')
    ax.grid(True, alpha=0.3)
    png_path = os.path.join(log_dir, f'section21_top20_singular_values_epoch{epoch}.png')
    fig.savefig(png_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    return png_path


def run_section_21_diagnostics(config_file, sweep_results):
    all_rows = []
    for item in sweep_results:
        overrides = {
            'train': {
                'subset_n': item['subset_n'],
                'subset_seed': item['subset_seed'],
                'batch_size': item['batch_size'],
            }
        }
        args = build_args(
            config_file=config_file,
            overrides=overrides,
            data_dir=DATA_DIR,
            log_dir='/tmp/diagnostics_logs',
            ckpt_dir='/tmp/diagnostics_ckpts',
            device='cuda' if torch.cuda.is_available() else 'cpu',
            download=False,
            create_dirs=False,
        )
        # Same shared N-pool loader for every checkpoint of this pool size.
        selected_indices = selected_indices_for_result(args, item)
        loader = build_diagnostics_loader(args, indices=selected_indices, batch_size=item['batch_size'])
        n_samples = len(loader.dataset) if item['subset_n'] is None else item['subset_n']
        log_dir = item.get('log_dir')
        if log_dir:
            os.makedirs(log_dir, exist_ok=True)

        for ckpt in checkpoints_for_result(item):
            epoch = epoch_from_checkpoint(ckpt)
            backbone = get_backbone(args.model.backbone).to(args.device)
            load_backbone_weights(backbone, ckpt)
            features, labels = extract_features(backbone, loader, args.device, n_samples=n_samples)
            singular_values, erank, explained_variance = spectral_diagnostics(features)
            n_train = int(0.8 * len(features))
            section21_knn = knn_eval(features, labels, k=min(20, n_train), n_train=n_train)
            row = {
                'N': item['N'],
                'epoch': epoch,
                'effective_rank': erank,
                'knn_accuracy': section21_knn,
                'monitor_accuracy': item.get('monitor_accuracy'),
                'checkpoint': ckpt,
            }
            all_rows.append(row)

            # Save THIS checkpoint's own diagnostics as separate files (one per checkpoint),
            # inside this pool size's log_dir, with the epoch in the filename.
            if log_dir:
                csv_path = os.path.join(log_dir, f'section21_diagnostics_epoch{epoch}.csv')
                pd.DataFrame([row]).to_csv(csv_path, index=False)
                png_path = save_checkpoint_figure(item['N'], epoch, singular_values[:20], log_dir)
                print(f"N={item['N']} epoch={epoch}: saved {csv_path} and {png_path}")

    return pd.DataFrame(all_rows)


diagnostics_table = run_section_21_diagnostics(CONFIG_FILE, results)
display(diagnostics_table)

# Optionally log the per-checkpoint diagnostics to Weights & Biases.
if globals().get('USE_WANDB', False):
    from analysis.wandb_logging import log_section21_diagnostics
    log_section21_diagnostics(
        diagnostics_table,
        project=globals().get('WANDB_PROJECT', 'instance_self_sup'),
        entity=globals().get('WANDB_ENTITY', None),
        run_name=f"{CONFIG_FILE.split('/')[-1].replace('.yaml', '')}-section21",
    )
